# 00 — Context: Decoherence-Free Sensing

**Core statement**

Decoherence-free subspaces specify robust differential quantum sensing.

This notebook establishes the baseline context for the `decoherence-free-sensing` repository. It does not try to reproduce the full Physical Review X paper. Instead, it builds the first computational layer for the lab report:

> common-mode noise → decoherence-free subspace → differential phase survives → entanglement improves scaling

**Reference paper**

Raphael Kaubruegger et al., *Lieb-Mattis States for Robust Entangled Differential Phase Sensing*, Physical Review X **16**, 021052 (2026). DOI: `10.1103/ksyh-mb4s`

## Runtime requirements

```bash
pip install numpy pandas matplotlib scipy ipython
```

In [ ]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle, FancyArrowPatch
from IPython.display import FileLink, display

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------
OUT = Path("outputs/notebook_00_context")
FIG = OUT / "figures"
DATA = OUT / "data"
SITE = OUT / "site"
for p in [FIG, DATA, SITE]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Plot helpers: no special styles; keep repository portable.
# ------------------------------------------------------------
def clean_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

def savefig(fig, name):
    path = FIG / name
    fig.savefig(path, dpi=180, bbox_inches="tight")
    return path

print(f"Writing outputs to: {OUT.resolve()}")

## 00 Paper map

The paper’s design chain can be read as a constraint-first engineering statement.

1. Two sensor nodes acquire phases \(\phi_A\) and \(\phi_B\).
2. Common-mode phase noise limits ordinary entangled sensors.
3. A decoherence-free subspace rejects the common phase.
4. Lieb-Mattis-type entanglement preserves differential sensitivity.
5. Cavity-mediated protocols provide practical preparation routes.

In [ ]:
paper_map = pd.DataFrame([
    ["sensor architecture", "two atomic ensembles", "estimate a differential phase"],
    ["dominant limitation", "common-mode phase noise", "shared noise corrupts ordinary readout"],
    ["leading specification", "decoherence-free subspace", "reject the common phase before optimizing entanglement"],
    ["target state family", "Lieb-Mattis / singlet-like DFS states", "retain high differential sensitivity"],
    ["measurement", "two-body DFS-compatible observable", "read out differential phase without leaving the DFS"],
    ["preparation", "two-mode squeezing or collective emission", "use cavity physics to generate useful entanglement"],
    ["engineering goal", "robust scalable quantum sensing", "go beyond SQL with realistic noise"],
], columns=["layer", "paper object", "role"])

paper_map.to_csv(DATA / "paper_map.csv", index=False)
paper_map.to_json(DATA / "paper_map.json", orient="records", indent=2)
paper_map

## 07 Leading specification flow

The leading specification is not merely the entangled state. The state is built to satisfy the deeper constraint:

> reject shared phase noise while preserving differential phase information.

In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 4.8))
clean_axes(ax)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Leading Specification Flow", fontsize=16, pad=16)

items = [
    ("Common-mode\nnoise", "same phase error\nat both nodes"),
    ("DFS constraint", "fixed J⁺z sector\nshared phase becomes global"),
    ("DFS entanglement", "Lieb-Mattis state\nor preparation proxy"),
    ("Differential\nphase sensing", "signal survives;\nnoise rejected"),
]
xs = np.linspace(0.13, 0.87, len(items))
for i, ((title, subtitle), x) in enumerate(zip(items, xs)):
    box = FancyBboxPatch((x-0.105, 0.40), 0.21, 0.30,
                         boxstyle="round,pad=0.02,rounding_size=0.03",
                         linewidth=1.5, facecolor="white", edgecolor="black")
    ax.add_patch(box)
    ax.text(x, 0.59, title, ha="center", va="center", fontsize=12, weight="bold")
    ax.text(x, 0.48, subtitle, ha="center", va="center", fontsize=9)
    if i < len(xs)-1:
        ax.add_patch(FancyArrowPatch((x+0.115, 0.55), (xs[i+1]-0.115, 0.55),
                                     arrowstyle="-|>", mutation_scale=16, linewidth=1.4))

ax.text(0.5, 0.18,
        "Constraint first: remove the common phase Φ, then optimize sensitivity to the differential phase φ.",
        ha="center", fontsize=11)
path_flow = savefig(fig, "00_leading_specification_flow.png")
plt.show()
path_flow

## 13 Phase variables

The two-node interferometer separates phase into common and differential coordinates:

\[
\Phi = \frac{\phi_A + \phi_B}{2}, \qquad
\varphi = \frac{\phi_A - \phi_B}{2}.
\]

A robust differential sensor should be insensitive to \(\Phi\) while remaining sensitive to \(\varphi\).

In [ ]:
phi_A = np.linspace(-np.pi, np.pi, 180)
phi_B = np.linspace(-np.pi, np.pi, 180)
A, B = np.meshgrid(phi_A, phi_B)
Phi = (A + B) / 2
varphi = (A - B) / 2

phase_grid = pd.DataFrame({
    "phi_A": A.ravel(),
    "phi_B": B.ravel(),
    "common_phase_Phi": Phi.ravel(),
    "differential_phase_varphi": varphi.ravel(),
})
phase_grid.sample(8, random_state=1)

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 6.2))
# Draw constant common and differential phase lines.
for c in np.linspace(-np.pi, np.pi, 9):
    # Phi = c => phi_B = 2c - phi_A
    ax.plot(phi_A, 2*c - phi_A, linewidth=0.8, alpha=0.55)
for d in np.linspace(-np.pi, np.pi, 9):
    # varphi = d => phi_B = phi_A - 2d
    ax.plot(phi_A, phi_A - 2*d, linewidth=0.8, alpha=0.55, linestyle="--")

ax.set_xlim(-np.pi, np.pi)
ax.set_ylim(-np.pi, np.pi)
ax.set_xlabel(r"node A phase $\phi_A$")
ax.set_ylabel(r"node B phase $\phi_B$")
ax.set_title("Common and Differential Phase Coordinates")
ax.text(-2.9, 2.55, "solid: constant common phase Φ", fontsize=10)
ax.text(-2.9, 2.25, "dashed: constant differential phase φ", fontsize=10)
ax.grid(True, alpha=0.25)
path_phase = savefig(fig, "13_phase_coordinates.png")
plt.show()
path_phase

## 17 Common-mode cancellation toy model

A simple toy model makes the design intent visible. Each node receives the same random phase noise \(\eta\), but the useful differential phase enters with opposite signs:

\[
\phi_A^{obs} = \eta + \varphi + \epsilon_A,\qquad
\phi_B^{obs} = \eta - \varphi + \epsilon_B.
\]

Subtracting the two node phases cancels \(\eta\). A DFS implements the quantum version of this cancellation structurally, before the readout step.

In [ ]:
rng = np.random.default_rng(42)
trials = 900
true_varphi = 0.21
common_noise = rng.normal(0, 1.20, trials)
readout_noise = rng.normal(0, 0.055, (trials, 2))

phi_A_obs = common_noise + true_varphi + readout_noise[:, 0]
phi_B_obs = common_noise - true_varphi + readout_noise[:, 1]
est_common = (phi_A_obs + phi_B_obs) / 2
est_diff = (phi_A_obs - phi_B_obs) / 2

noise_summary = pd.DataFrame({
    "quantity": ["true differential phase", "std common estimate", "std differential estimate", "mean differential estimate"],
    "value": [true_varphi, np.std(est_common), np.std(est_diff), np.mean(est_diff)]
})
noise_summary.to_csv(DATA / "toy_common_mode_summary.csv", index=False)
noise_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 5.8))
ax.scatter(phi_A_obs, phi_B_obs, s=12, alpha=0.35)
ax.set_xlabel(r"observed node A phase")
ax.set_ylabel(r"observed node B phase")
ax.set_title("Common-Mode Noise Moves Both Nodes Together")
ax.grid(True, alpha=0.25)
ax.text(0.05, 0.95,
        "points move along shared-noise diagonal\nseparation still estimates differential phase",
        transform=ax.transAxes, va="top", fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", edgecolor="black", alpha=0.9))
path_noise_cloud = savefig(fig, "17_common_mode_noise_cloud.png")
plt.show()
path_noise_cloud

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 4.8))
ax.hist(est_common, bins=40, alpha=0.65, label="common estimate")
ax.hist(est_diff, bins=40, alpha=0.65, label="differential estimate")
ax.axvline(true_varphi, linestyle="--", linewidth=1.5, label="true differential phase")
ax.set_title("Differential Estimate Rejects Shared Noise")
ax.set_xlabel("phase estimate")
ax.set_ylabel("count")
ax.legend()
path_noise_hist = savefig(fig, "17_common_mode_histogram.png")
plt.show()
path_noise_hist

## 23 DFS basis count

For two equal ensembles in a symmetric Dicke representation, a DFS with fixed \(J_z^+\) contains basis states whose two node magnetizations sum to a constant.

Notebook 00 only counts this reduced state space. Later notebooks can build explicit matrix representations.

In [ ]:
def dfs_basis_rows(N_total):
    # Symmetric Dicke labels for two equal ensembles of N_total/2 atoms each.
    # The collective spin in each node is j = N_total/4.  We keep rows with
    # mA + mB = 0, corresponding to the central DFS used by the target state.
    if N_total % 4 != 0:
        raise ValueError("Use N_total divisible by 4 so each node has integer j=N/4.")
    j = N_total // 4
    rows = []
    for mA in range(-j, j + 1):
        mB = -mA
        rows.append({"N_total": N_total, "j_node": j, "mA": mA, "mB": mB, "Jz_plus": mA + mB, "Jz_minus": mA - mB})
    return pd.DataFrame(rows)

basis_examples = pd.concat([dfs_basis_rows(N) for N in [8, 16, 32, 64]], ignore_index=True)
basis_examples.to_csv(DATA / "dfs_basis_examples.csv", index=False)
basis_examples.head(12)

In [ ]:
Ns_basis = np.arange(4, 401, 4)
dims = np.array([len(dfs_basis_rows(int(N))) for N in Ns_basis])

fig, ax = plt.subplots(figsize=(7.8, 5.0))
ax.plot(Ns_basis, dims, marker="o", markersize=3, linewidth=1.5)
ax.set_xlabel("total atom number N")
ax.set_ylabel("central DFS dimension in symmetric basis")
ax.set_title("Central DFS Dimension Grows Linearly in Symmetric Representation")
ax.grid(True, alpha=0.25)
path_dim = savefig(fig, "23_dfs_dimension_scaling.png")
plt.show()
path_dim

## 29 SQL, Heisenberg, and Lieb-Mattis QFI references

The target Lieb-Mattis state has quantum Fisher information

\[
F_Q = \frac{N^2 + 4N}{3}.
\]

The corresponding one-shot QCRB reference variance is

\[
\Delta^2 \varphi \geq \frac{1}{F_Q}.
\]

This notebook uses that as a scaling reference, not as a full noise simulation.

In [ ]:
N = np.arange(4, 2001, 4)
FQ_lieb_mattis = (N**2 + 4*N) / 3
variance_lieb_mattis = 1 / FQ_lieb_mattis
sql = 1 / N
heisenberg = 1 / N**2

scaling = pd.DataFrame({
    "N": N,
    "SQL_variance_1_over_N": sql,
    "Heisenberg_variance_1_over_N2": heisenberg,
    "Lieb_Mattis_QFI": FQ_lieb_mattis,
    "Lieb_Mattis_QCRB_variance": variance_lieb_mattis,
    "LM_improvement_over_SQL": sql / variance_lieb_mattis,
    "LM_factor_above_HL": variance_lieb_mattis / heisenberg,
})
scaling.to_csv(DATA / "qfi_scaling_reference.csv", index=False)
scaling.to_json(DATA / "qfi_scaling_reference.json", orient="records")
scaling.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8.0, 5.6))
ax.loglog(N, sql, label="SQL: 1/N", linewidth=2)
ax.loglog(N, heisenberg, label="Heisenberg: 1/N²", linewidth=2)
ax.loglog(N, variance_lieb_mattis, label="Lieb-Mattis QCRB: 3/(N²+4N)", linewidth=2)
ax.set_xlabel("total atom number N")
ax.set_ylabel(r"variance reference $\Delta^2\varphi$")
ax.set_title("Scaling References for Differential Quantum Sensing")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
path_scaling = savefig(fig, "29_sql_hl_lieb_mattis_scaling.png")
plt.show()
path_scaling

## 37 Two-body fringe response

For the target state, the paper gives the two-body measurement response

\[
\langle M_\varphi \rangle = -\frac{N^2+4N}{12}\cos(2\varphi).
\]

The doubled phase dependence appears because the measurement is a two-body observable compatible with the DFS.

In [ ]:
phis = np.linspace(0, np.pi/2, 500)
Ns_fringe = [40, 120, 320]
fringe_rows = []

fig, ax = plt.subplots(figsize=(8.2, 5.2))
for N0 in Ns_fringe:
    amp = (N0**2 + 4*N0) / 12
    signal = -amp * np.cos(2*phis)
    ax.plot(phis, signal / amp, linewidth=2, label=f"N={N0}")
    for p, s in zip(phis[::50], signal[::50]):
        fringe_rows.append({"N": N0, "varphi": p, "normalized_signal": s/amp, "raw_signal": s})

ax.axvline(np.pi/4, linestyle="--", linewidth=1.4, label="zero crossing π/4")
ax.set_xlabel(r"differential phase $\varphi$")
ax.set_ylabel(r"normalized $\langle M_\varphi\rangle$")
ax.set_title("DFS-Compatible Two-Body Fringe Response")
ax.grid(True, alpha=0.25)
ax.legend()
path_fringe = savefig(fig, "37_two_body_fringe_response.png")
plt.show()

fringes = pd.DataFrame(fringe_rows)
fringes.to_csv(DATA / "two_body_fringe_samples.csv", index=False)
path_fringe

## 43 Preparation protocol map

Notebook 00 only maps the preparation design space. Later notebooks can simulate Hamiltonian or Lindblad dynamics.

In [ ]:
protocols = pd.DataFrame([
    ["adiabatic unitary", "sweep field gradient", "target Lieb-Mattis state", "high fidelity in principle", "challenging timescale"],
    ["two-mode squeezing quench", "Hamiltonian proxy", "DFS entangled proxy", "Heisenberg scaling reference", "optimize quench time and detuning"],
    ["dissipative collective emission", "shared cavity jump process", "mixed Lieb-Mattis-like state", "robust / present-day compatible", "sqrt improvement beyond SQL"],
    ["photon-count conditioning", "read emitted photons", "postselected lower-J states", "can improve variance", "depends on preparation vs interrogation time"],
], columns=["protocol", "mechanism", "prepared object", "strength", "constraint"])

protocols.to_csv(DATA / "preparation_protocol_map.csv", index=False)
protocols.to_json(DATA / "preparation_protocol_map.json", orient="records", indent=2)
protocols

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.1))
clean_axes(ax)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title("Preparation Protocol Map", fontsize=16, pad=16)

cols = [
    ("Unitary", "two-mode squeezing\nquench proxy", "fast entangling\ndynamics"),
    ("Dissipative", "collective emission\ninto cavity", "robust mixed\nDFS state"),
    ("Conditioned", "photon count\nselection", "restart if too\nmany photons"),
]
xs = [0.18, 0.50, 0.82]
for x, (title, mech, note) in zip(xs, cols):
    ax.add_patch(FancyBboxPatch((x-0.13, 0.30), 0.26, 0.42,
                                boxstyle="round,pad=0.025,rounding_size=0.03",
                                facecolor="white", edgecolor="black", linewidth=1.4))
    ax.text(x, 0.62, title, ha="center", fontsize=13, weight="bold")
    ax.text(x, 0.49, mech, ha="center", fontsize=10)
    ax.text(x, 0.37, note, ha="center", fontsize=10)

for x1, x2 in [(0.31,0.37),(0.63,0.69)]:
    ax.add_patch(FancyArrowPatch((x1,0.51),(x2,0.51), arrowstyle="-|>", mutation_scale=16, linewidth=1.2))
ax.text(0.5, 0.16, "All routes are judged by the same specification: stay robust to shared noise while preserving differential sensitivity.",
        ha="center", fontsize=11)
path_protocols = savefig(fig, "43_preparation_protocol_map.png")
plt.show()
path_protocols

## 53 Sensor design table

This table turns the paper context into repository tasks.

In [ ]:
design_rules = pd.DataFrame([
    ["Reject common phase first", "operate inside a DFS of Jz+", "07_dfs_constraint"],
    ["Use DFS-compatible observables", "measure two-body operators that preserve the DFS", "11_measurement_operator"],
    ["Construct the target state", "build Lieb-Mattis amplitudes in the Dicke basis", "13_lieb_mattis_state"],
    ["Compare scaling references", "SQL, HL, Lieb-Mattis QFI", "17_qfi_scaling"],
    ["Model unitary preparation", "two-mode squeezing quench", "23_two_mode_squeezing"],
    ["Model dissipative preparation", "collective emission into cavity", "29_collective_emission"],
    ["Track realistic noise", "cooperativity, free-space emission, particle number", "37_noise_scaling"],
    ["Extract engineering statements", "constraints and admissible operating regimes", "43_design_rules"],
], columns=["design rule", "computational expression", "planned notebook"])

design_rules.to_csv(DATA / "sensor_design_rules.csv", index=False)
design_rules.to_json(DATA / "sensor_design_rules.json", orient="records", indent=2)
design_rules

## 67 Repo roadmap

In [ ]:
roadmap = pd.DataFrame([
    ["00_context", "common-mode noise, DFS constraint, scaling references", "complete"],
    ["07_dfs_constraint", "show why Jz+ eigenstates reject common phase", "next"],
    ["11_measurement_operator", "construct DFS-compatible two-body observable M", "planned"],
    ["13_lieb_mattis_state", "construct target-state amplitudes and singlet interpretation", "planned"],
    ["17_qfi_scaling", "derive/verify QFI references and QCRB plots", "planned"],
    ["23_two_mode_squeezing", "simulate quench proxy and optimize time", "planned"],
    ["29_collective_emission", "simulate dissipative preparation map", "planned"],
    ["37_noise_scaling", "cooperativity and free-space emission scaling", "planned"],
    ["43_design_rules", "engineering statement extraction", "planned"],
], columns=["notebook", "purpose", "status"])

roadmap.to_csv(DATA / "repo_roadmap.csv", index=False)
roadmap.to_json(DATA / "repo_roadmap.json", orient="records", indent=2)
roadmap

## 71 Small site index

This writes a minimal `site/index.html` summary that can be copied into docs or GitHub Pages later.

In [ ]:
site_html = r'''<!doctype html>
<html lang=\"en\">
<head>
  <meta charset=\"utf-8\">
  <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\">
  <title>Decoherence-Free Sensing — Notebook 00</title>
</head>
<body>
  <main>
    <h1>Decoherence-Free Subspaces Specify Robust Quantum Sensing</h1>
    <p><strong>Notebook 00:</strong> Context, phase variables, DFS constraint, and scaling references.</p>
    <p>Core statement: reject common-mode phase noise first, then engineer entanglement for the remaining differential signal.</p>
    <ul>
      <li>Common/differential phase coordinates</li>
      <li>DFS basis count</li>
      <li>SQL, Heisenberg, and Lieb-Mattis QFI references</li>
      <li>Preparation protocol map</li>
    </ul>
  </main>
</body>
</html>
'''
(SITE / "index.html").write_text(site_html)
SITE / "index.html"

## 79 Export summary metadata

In [ ]:
summary = {
    "notebook": "00_context",
    "repo": "decoherence-free-sensing",
    "title": "Decoherence-Free Subspaces Specify Robust Quantum Sensing",
    "paper": "Lieb-Mattis States for Robust Entangled Differential Phase Sensing",
    "doi": "10.1103/ksyh-mb4s",
    "core_statement": "Reject common-mode phase noise first, then engineer entanglement for differential sensitivity.",
    "leading_specification": "Decoherence-free subspaces remove common-mode phase noise while preserving differential phase signals.",
    "generated_figures": sorted([p.name for p in FIG.glob("*.png")]),
    "generated_data": sorted([p.name for p in DATA.glob("*")]),
    "next_notebook": "07_dfs_constraint",
}

summary_path = OUT / "context_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

summary_df = pd.DataFrame([summary | {
    "generated_figures": ", ".join(summary["generated_figures"]),
    "generated_data": ", ".join(summary["generated_data"]),
}])
summary_df.to_csv(OUT / "context_summary.csv", index=False)
summary

## 89 Package and download outputs

Run this final cell after executing the notebook. It creates a zip containing all generated figures, tables, JSON summaries, and the small site stub.

In [ ]:
# ============================================================
# Package and download Notebook 00 outputs
# ============================================================
#
# This cell mirrors the download pattern used in other repos:
#   - collect outputs under outputs/notebook_00_context
#   - create a zip
#   - display a clickable FileLink
#
# In Google Colab, uncomment the optional files.download line.

zip_base = Path("outputs/notebook_00_context")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=zip_base)
print(f"Created: {zip_path}")
display(FileLink(zip_path))

# Optional Colab download:
# from google.colab import files
# files.download(zip_path)